## CS-4063: Natural Language Processing — Assignment 2
## Neural NLP Pipeline (BBC Urdu Corpus)

| | |
|---|---|
| **Name** | Munaza Tariq |
| **Roll Number** | i23-2545 |
| **Section** | DS-A |
| **GitHub** | [i23-2545-NLP-Assignment2](https://github.com/i23-2545/i23-2545-NLP-Assignment2) |

---

In [ ]:
import os, re, json, math, random, warnings
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import rcParams

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.manifold import TSNE
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, confusion_matrix, classification_report)
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

# --- Reproducibility -------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# --- Paths -----------------------------------------------------------------
BASE_DIR   = Path('.')
DATA_DIR   = BASE_DIR / 'data'
EMB_DIR    = BASE_DIR / 'embeddings'
MODEL_DIR  = BASE_DIR / 'models'
EMB_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__}  |  Device: {DEVICE}')

## 0. Data Loading Utilities

Load `cleaned.txt`, `raw.txt`, and `Metadata.json` produced by Assignment 1. 
Each article in the `.txt` files is delimited by `[N]` markers.

In [ ]:
# ---------------------------------------------------------------------------
# 0.1  Load text corpus
# ---------------------------------------------------------------------------
def load_corpus(filepath: str) -> list[list[str]]:
    """Load a .txt corpus file into a list of articles.
    
    Each article is a list of sentences (strings).
    Articles are separated by lines matching `[N]`.
    """
    articles: list[list[str]] = []
    current_article: list[str] = []
    
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            # Article delimiter: [1], [2], ...
            if re.match(r'^\[\d+\]$', line):
                if current_article:
                    articles.append(current_article)
                current_article = []
            else:
                current_article.append(line)
    # Don't forget the last article
    if current_article:
        articles.append(current_article)
    
    return articles


def tokenize_article(sentences: list[str]) -> list[str]:
    """Simple whitespace tokenizer for an article's sentences."""
    tokens = []
    for sent in sentences:
        tokens.extend(sent.split())
    return tokens


# Load both corpora
cleaned_articles = load_corpus(DATA_DIR / 'cleaned.txt')
raw_articles     = load_corpus(DATA_DIR / 'raw.txt')

print(f'Cleaned corpus: {len(cleaned_articles)} articles')
print(f'Raw corpus:     {len(raw_articles)} articles')
print(f'Sample cleaned article (first 3 sentences):')
for s in cleaned_articles[0][:3]:
    print(f'  {s}')

In [ ]:
# ---------------------------------------------------------------------------
# 0.2  Load Metadata
# ---------------------------------------------------------------------------
with open(DATA_DIR / 'Metadata.json', 'r', encoding='utf-8') as f:
    metadata = json.load(f)

print(f'Metadata entries: {len(metadata)}')
# Show first 5 entries
for k in list(metadata.keys())[:5]:
    print(f'  [{k}] {metadata[k]["title"][:60]}...')

### Vocabulary Builder

Build a vocabulary from the **cleaned** corpus, keeping the **top 10,000 most frequent tokens**. 
All other tokens are mapped to `<UNK>`. We also add a `<PAD>` token for padding sequences.

In [ ]:
# ---------------------------------------------------------------------------
# 0.3  Build Vocabulary
# ---------------------------------------------------------------------------
VOCAB_SIZE = 10000  # top-k tokens
SPECIAL_TOKENS = ['<PAD>', '<UNK>', '<CLS>']

def build_vocabulary(articles: list[list[str]], top_k: int = VOCAB_SIZE):
    """Build word2idx and idx2word mappings from the corpus.
    
    Returns:
        word2idx: dict mapping token -> integer index
        idx2word: list mapping index -> token
        token_counts: Counter of all token frequencies
    """
    # Count all tokens across all articles
    token_counts = Counter()
    for art in articles:
        for sent in art:
            token_counts.update(sent.split())
    
    # Select top-k most frequent tokens
    most_common = token_counts.most_common(top_k)
    
    # Build mappings with special tokens first
    idx2word = list(SPECIAL_TOKENS)
    for token, _ in most_common:
        if token not in SPECIAL_TOKENS:
            idx2word.append(token)
    
    word2idx = {w: i for i, w in enumerate(idx2word)}
    
    return word2idx, idx2word, token_counts


word2idx, idx2word, token_counts = build_vocabulary(cleaned_articles)

PAD_IDX = word2idx['<PAD>']
UNK_IDX = word2idx['<UNK>']
CLS_IDX = word2idx['<CLS>']

print(f'Total unique tokens in corpus: {len(token_counts):,}')
print(f'Vocabulary size (with special): {len(word2idx):,}')
print(f'Coverage: top {VOCAB_SIZE} tokens cover '
      f'{sum(c for _, c in token_counts.most_common(VOCAB_SIZE)):,} / '
      f'{sum(token_counts.values()):,} '
      f'({100*sum(c for _, c in token_counts.most_common(VOCAB_SIZE))/sum(token_counts.values()):.1f}%) occurrences')
print(f'\nTop 20 tokens: {[t for t, _ in token_counts.most_common(20)]}')

In [ ]:
# ---------------------------------------------------------------------------
# 0.4  Helper: encode tokens to indices
# ---------------------------------------------------------------------------
def tokens_to_ids(tokens: list[str], w2i: dict) -> list[int]:
    """Convert a list of tokens to a list of integer IDs."""
    unk = w2i['<UNK>']
    return [w2i.get(t, unk) for t in tokens]


def article_to_ids(article: list[str], w2i: dict) -> list[int]:
    """Tokenize and encode an entire article."""
    tokens = tokenize_article(article)
    return tokens_to_ids(tokens, w2i)


# Quick sanity check
sample_ids = article_to_ids(cleaned_articles[0], word2idx)
print(f'Article 0: {len(sample_ids)} tokens')
print(f'First 20 IDs: {sample_ids[:20]}')
print(f'Decoded back: {[idx2word[i] for i in sample_ids[:20]]}')

In [ ]:
# ---------------------------------------------------------------------------
# 0.5  Topic Category Assignment (keyword-based)
# ---------------------------------------------------------------------------
# We assign each article to one of 5 topic categories using indicative
# keywords from Metadata.json titles (in Urdu).

TOPIC_KEYWORDS = {
    'Politics': [
        'حکومت', 'وزیر', 'اعلیٰ', 'پارلیمنٹ', 'انتخاب', 'سیاسی', 'جمہوریت',
        'قومی', 'اسمبلی', 'صدر', 'وزیراعظم', 'حزب', 'اختلاف', 'عدالت',
        'سپریم', 'کورٹ', 'آئین', 'قانون', 'فیصلہ', 'گرفتار', 'مقدمہ',
        'پولیس', 'دھماکہ', 'حملہ', 'دہشت', 'بم', 'فائرنگ', 'ہلاک',
        'سکیورٹی', 'فوج', 'آپریشن', 'طالبان', 'بلوچستان', 'مسلح',
        'حملے', 'شدت', 'دفاع', 'امن', 'خودکش'
    ],
    'Sports': [
        'کرکٹ', 'میچ', 'ٹیم', 'کھلاڑی', 'ورلڈ', 'کپ', 'فٹبال', 'اسکور',
        'بولنگ', 'بیٹنگ', 'سیریز', 'ٹورنامنٹ', 'فائنل', 'سیمی', 'اوور',
        'رنز', 'وکٹ', 'گول', 'لیگ', 'رونالڈو', 'پاکستان کرکٹ',
        'آئی سی سی', 'کھیل', 'سٹیڈیم', 'اننگز', 'کپتان', 'ایونٹ',
        'سپن', 'بلے', 'باز', 'فاسٹ', 'ریکارڈ'
    ],
    'Economy': [
        'معیشت', 'تجارت', 'بینک', 'بجٹ', 'مہنگائی', 'روپے', 'ڈالر',
        'سرمایہ', 'ٹیکس', 'آمدنی', 'شرح', 'سود', 'قرض', 'پیسے',
        'معاشی', 'اقتصادی', 'مارکیٹ', 'ملازمت', 'تنخواہ', 'مالی',
        'بے روزگاری', 'سرمایہ کاری', 'برآمد', 'درآمد'
    ],
    'International': [
        'امریکہ', 'ٹرمپ', 'چین', 'بھارت', 'انڈیا', 'روس', 'برطانیہ',
        'یورپ', 'اقوام', 'متحدہ', 'سفارتی', 'بین الاقوامی', 'جنگ',
        'غزہ', 'اسرائیل', 'فلسطین', 'یوکرین', 'ناٹو', 'ایران',
        'افغانستان', 'سعودی', 'ترکی', 'عرب', 'مشرق وسطیٰ', 'سفیر',
        'ناروے', 'جرمنی', 'فرانس', 'جاپان', 'کینیڈا', 'آسٹریلیا',
        'دنیا', 'ممالک', 'مغربی', 'سپین'
    ],
    'Health & Society': [
        'ہسپتال', 'بیماری', 'ویکسین', 'صحت', 'تعلیم', 'سکول', 'یونیورسٹی',
        'خواتین', 'بچے', 'آبادی', 'سیلاب', 'زلزلہ', 'موسم', 'ماحول',
        'طوفان', 'آلودگی', 'پانی', 'ادویات', 'ڈاکٹر', 'مریض', 'علاج',
        'سماجی', 'معاشرہ', 'ثقافت', 'تہوار', 'مذہب', 'شادی', 'رسم',
        'پتنگ', 'بسنت', 'عورت', 'لڑکی', 'استحصال', 'جنسی'
    ]
}

def classify_article(article_id: str, meta: dict, articles: list) -> str:
    """Classify an article into one of 5 topic categories.
    
    Uses both the title from metadata AND the article text for matching.
    """
    idx = int(article_id) - 1
    title = meta.get(article_id, {}).get('title', '')
    
    # Combine title + first 5 sentences for richer signal
    text = title
    if 0 <= idx < len(articles):
        text += ' ' + ' '.join(articles[idx][:5])
    
    scores = {}
    for category, keywords in TOPIC_KEYWORDS.items():
        score = sum(1 for kw in keywords if kw in text)
        scores[category] = score
    
    best = max(scores, key=scores.get)
    # If no keywords matched, default to 'Health & Society'
    if scores[best] == 0:
        best = 'Health & Society'
    return best


# Classify all articles
article_topics = {}
for art_id in metadata:
    article_topics[art_id] = classify_article(art_id, metadata, cleaned_articles)

# Report distribution
topic_dist = Counter(article_topics.values())
print('\n=== Topic Distribution ===')
for topic, count in topic_dist.most_common():
    print(f'  {topic:25s}: {count:3d} articles ({100*count/len(article_topics):.1f}%)')
print(f'  {"TOTAL":25s}: {len(article_topics):3d}')

In [ ]:
# ---------------------------------------------------------------------------
# 0.6  Save vocabulary (word2idx.json)
# ---------------------------------------------------------------------------
vocab_path = EMB_DIR / 'word2idx.json'
with open(vocab_path, 'w', encoding='utf-8') as f:
    json.dump(word2idx, f, ensure_ascii=False, indent=2)

print(f'Saved vocabulary ({len(word2idx):,} entries) to {vocab_path}')

# Also save topic labels for later use
topics_path = DATA_DIR / 'article_topics.json'
with open(topics_path, 'w', encoding='utf-8') as f:
    json.dump(article_topics, f, ensure_ascii=False, indent=2)

print(f'Saved article topics to {topics_path}')